In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
import math
import random

try:
    from IPython.display import display, clear_output
except ImportError:
    pass

CONFIG = {
    "VOLUME_PATH": "sliepner.npy",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
    "EPOCHS": 500,              # Increased for convergence
    "LR": 1e-4,
    "LR_DECAY_STEP": 200,
    "SCALE": 2,
    "SAVE_DIR": "zssr_results",
    "PLOT_TRANSPOSE": True,
    "SEED": 42,
    "CLIP_PERCENTILE": 1.0,
    "CROP_SIZE": 64,
    "VISUALIZE_INTERVAL": 50,
    "FILTERS": 64,
    "CHANNELS": 1,
    "KERNEL_SIZE": 3,
    "NUM_BODY_LAYERS": 6,
    "DATA_RANGE": 2.0,
    "TEST_INDICES": [20, 50, 80],
    "BLUR_SIGMA": 1.0           # New: Controls the severity of degradation
}

os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)
torch.manual_seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
random.seed(CONFIG["SEED"])

def transpose_and_plot(ax, img, title):
    if isinstance(img, torch.Tensor):
        arr = img.squeeze().detach().cpu().numpy()
    else:
        arr = np.array(img)
    if CONFIG["PLOT_TRANSPOSE"]:
        arr = arr.T
    
    robust_max = np.percentile(np.abs(arr), 100 - CONFIG["CLIP_PERCENTILE"])
    ax.imshow(arr, cmap="RdBu", aspect="auto", vmin=-robust_max, vmax=robust_max)
    ax.set_title(title)
    ax.axis("off")

def compute_metrics(sr, hr, data_range=None):
    if data_range is None:
        data_range = CONFIG["DATA_RANGE"]
    sr_np = sr.detach().cpu().squeeze().numpy()
    hr_np = hr.detach().cpu().squeeze().numpy()
    
    psnr = psnr_metric(hr_np, sr_np, data_range=data_range)
    ssim = ssim_metric(hr_np, sr_np, data_range=data_range)
    mae = np.mean(np.abs(sr_np - hr_np))
    return psnr, ssim, mae

def data_augmentation(image, mode):
    if mode == 0: return image
    if mode == 1: return np.rot90(image, k=1, axes=(-2, -1)).copy()
    if mode == 2: return np.rot90(image, k=2, axes=(-2, -1)).copy()
    if mode == 3: return np.rot90(image, k=3, axes=(-2, -1)).copy()
    if mode == 4: return np.flip(image, axis=-2).copy()
    if mode == 5: return np.flip(np.rot90(image, k=1, axes=(-2, -1)), axis=-2).copy()
    if mode == 6: return np.flip(np.rot90(image, k=2, axes=(-2, -1)), axis=-2).copy()
    if mode == 7: return np.flip(np.rot90(image, k=3, axes=(-2, -1)), axis=-2).copy()

def get_blur_kernel(sigma, device):
    """Creates a Gaussian blur kernel to mimic realistic seismic degradation."""
    k_size = int(math.ceil(2 * sigma)) * 2 + 1
    k_half = (k_size - 1) * 0.5
    
    x = torch.linspace(-k_half, k_half, steps=k_size).to(device)
    pdf = torch.exp(-0.5 * (x / sigma).pow(2))
    kernel1d = pdf / pdf.sum()
    
    kernel2d = kernel1d.unsqueeze(1) @ kernel1d.unsqueeze(0)
    kernel2d = kernel2d.unsqueeze(0).unsqueeze(0) # [1, 1, k, k]
    return kernel2d

def apply_degradation(img, kernel, scale):
    """
    Applies Blur -> Downsample. 
    This creates a harder task than simple interpolation.
    """
    # 1. Blur (Low-pass filter)
    # Reflect padding to avoid border artifacts
    pad = kernel.shape[-1] // 2
    blurred = F.conv2d(img, kernel, padding=pad)
    
    # 2. Downsample
    lr = F.interpolate(blurred, scale_factor=1/scale, mode='bicubic', align_corners=False)
    return lr

class ZSSRNet(nn.Module):
    def __init__(self):
        super(ZSSRNet, self).__init__()
        filters = CONFIG["FILTERS"]
        kernel_size = CONFIG["KERNEL_SIZE"]
        channels = CONFIG["CHANNELS"]
        
        self.head = nn.Sequential(
            nn.Conv2d(channels, filters, kernel_size=kernel_size, stride=1, padding=kernel_size//2),
            nn.ReLU(inplace=True)
        )
        
        self.body = nn.Sequential(
            *[nn.Sequential(
                nn.Conv2d(filters, filters, kernel_size=kernel_size, stride=1, padding=kernel_size//2),
                nn.ReLU(inplace=True)
            ) for _ in range(CONFIG["NUM_BODY_LAYERS"])]
        )
        
        self.tail = nn.Conv2d(filters, channels, kernel_size=kernel_size, stride=1, padding=kernel_size//2)

    def forward(self, x):
        res = self.head(x)
        res = self.body(res) + res 
        out = self.tail(res)
        return out + x 

def run_zssr_on_slice(hr_original_slice, slice_idx):
    print(f"\n--- Starting ZSSR for Slice {slice_idx} ---")
    
    scale_factor = np.percentile(np.abs(hr_original_slice), 99.5)
    if scale_factor == 0: scale_factor = 1.0
    hr_norm = hr_original_slice / scale_factor
    hr_norm = np.clip(hr_norm, -1.0, 1.0)
    
    h, w = hr_norm.shape
    
    hr_tensor = torch.tensor(hr_norm).float().unsqueeze(0).unsqueeze(0).to(CONFIG["DEVICE"])
    
    # --- DEGRADATION SETUP ---
    # We define a blur kernel. This will be used to generate the "Real" LR from HR
    # AND to generate "Son" patches from "Father" patches during training.
    blur_kernel = get_blur_kernel(CONFIG["BLUR_SIGMA"], CONFIG["DEVICE"])
    
    # Generate the Input LR image (Observation)
    lr_tensor = apply_degradation(hr_tensor, blur_kernel, CONFIG["SCALE"])
    lr_h, lr_w = lr_tensor.shape[2], lr_tensor.shape[3]
    
    # Baseline: Simple Upscale of the blurred input
    bicubic_baseline = F.interpolate(lr_tensor, size=(h, w), mode='bicubic', align_corners=False)
    
    model = ZSSRNet().to(CONFIG["DEVICE"])
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["LR"])
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=CONFIG["LR_DECAY_STEP"], gamma=0.5)
    criterion = nn.L1Loss()
    
    lr_img_np = lr_tensor.squeeze().cpu().numpy()
    
    pbar = tqdm(range(CONFIG["EPOCHS"]), desc="ZSSR Internal Learning")
    
    for epoch in pbar:
        model.train()
        optimizer.zero_grad()
        
        crop_size = CONFIG["CROP_SIZE"]
        if lr_h < crop_size or lr_w < crop_size: crop_size = min(lr_h, lr_w)
            
        x = np.random.randint(0, lr_h - crop_size + 1)
        y = np.random.randint(0, lr_w - crop_size + 1)
        
        father_crop = lr_img_np[x:x+crop_size, y:y+crop_size]
        
        aug_mode = np.random.randint(0, 8)
        father_crop_aug = data_augmentation(father_crop, aug_mode)
        
        father_tensor = torch.tensor(father_crop_aug).float().unsqueeze(0).unsqueeze(0).to(CONFIG["DEVICE"])
        
        # --- INTERNAL LEARNING DEGRADATION ---
        # Crucial: Apply the SAME degradation to Father -> Son
        # that we assume happened to HR -> LR
        son_tensor = apply_degradation(father_tensor, blur_kernel, CONFIG["SCALE"])
        
        # Upsample Son to match Father size (Network learns residual correction)
        son_interpolated = F.interpolate(son_tensor, size=(crop_size, crop_size), mode='bicubic', align_corners=False)
        
        pred_father = model(son_interpolated)
        
        loss = criterion(pred_father, father_tensor)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        pbar.set_postfix({'loss': f"{loss.item():.6f}"})
        
        if epoch % CONFIG["VISUALIZE_INTERVAL"] == 0 or epoch == CONFIG["EPOCHS"] - 1:
            model.eval()
            with torch.no_grad():
                try:
                    clear_output(wait=True)
                except NameError:
                    pass

                # Inference: Upsample LR -> SR
                lr_upsampled = F.interpolate(lr_tensor, size=(h, w), mode='bicubic', align_corners=False)
                sr_tensor = model(lr_upsampled)
                
                sr_denorm = sr_tensor * scale_factor
                hr_denorm = hr_tensor * scale_factor
                base_denorm = bicubic_baseline * scale_factor
                
                dr = hr_denorm.max().item() - hr_denorm.min().item()
                psnr_zssr, ssim_zssr, mae_zssr = compute_metrics(sr_denorm, hr_denorm, dr)
                psnr_base, ssim_base, mae_base = compute_metrics(base_denorm, hr_denorm, dr)
                
                print(f"Slice {slice_idx} | Epoch {epoch} | Base PSNR: {psnr_base:.2f} | ZSSR PSNR: {psnr_zssr:.2f}")

                try:
                    # Update: 2 Rows of plotting (Full Image + Training Patches)
                    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
                    
                    # Row 1: Full Image Inference
                    transpose_and_plot(axes[0, 0], base_denorm[0,0], f"Full Baseline (Bicubic)\nPSNR: {psnr_base:.2f} | MAE: {mae_base:.4f}")
                    transpose_and_plot(axes[0, 1], hr_denorm[0,0], f"Full Target (Slice {slice_idx})")
                    transpose_and_plot(axes[0, 2], sr_denorm[0,0], f"Full ZSSR (Unsupervised)\nPSNR: {psnr_zssr:.2f} | MAE: {mae_zssr:.4f}")
                    
                    # Row 2: Internal Training Patches (What the network sees)
                    transpose_and_plot(axes[1, 0], son_interpolated[0,0].detach().cpu(), "Training Input (Son)\n(Blurred+Down+Up)")
                    transpose_and_plot(axes[1, 1], father_tensor[0,0].detach().cpu(), "Training Target (Father)\n(HR Crop)")
                    transpose_and_plot(axes[1, 2], pred_father[0,0].detach().cpu(), "Training Prediction\n(Net Output)")
                    
                    plt.suptitle(f"ZSSR Training Step {epoch} (Sigma={CONFIG['BLUR_SIGMA']})")
                    plt.tight_layout()
                    plt.savefig(os.path.join(CONFIG["SAVE_DIR"], f"slice_{slice_idx}_step_{epoch}.png"))
                    plt.show()
                    plt.close()
                except Exception as e:
                    print(f"Plotting error: {e}")

    return psnr_zssr, psnr_base

def main():
    if not os.path.exists(CONFIG["VOLUME_PATH"]):
        print("Creating dummy volume...")
        vol = np.random.randn(100, 128, 128).astype(np.float32)
    else:
        vol = np.load(CONFIG["VOLUME_PATH"]).astype(np.float32)
        
    print(f"Volume Shape: {vol.shape}")
    
    test_indices = CONFIG["TEST_INDICES"]
    if vol.shape[0] < max(test_indices):
        test_indices = [0, vol.shape[0]//2]
    
    for idx in test_indices:
        slice_data = vol[idx]
        run_zssr_on_slice(slice_data, idx)

if __name__ == "__main__":
    main()